### Conversion Dataset a Parquet & Creación de Clase Ternaria

#### Configuración Recursos según Máquina

In [ ]:

entorno = 'casa'  # cambiar a 'oficina' según corresponda

if entorno == 'casa':
    # i7-12700K, 12 cores, 32GB RAM
    duckdb_threads = 8
    duckdb_memory_limit = '16GB'
    rf_n_jobs = 8
    lgb_threads = 8
elif entorno == 'oficina':
    # 16 cores, 63.8GB RAM
    duckdb_threads = 12
    duckdb_memory_limit = '45GB'
    rf_n_jobs = 10
    lgb_threads = 10          # 8 por precaución: la oficina se apagó con CPU al 100% el 27/09
else:
    raise ValueError("entorno debe ser 'casa' u 'oficina'")

print(f"Entorno: {entorno} | DuckDB: {duckdb_threads} threads, {duckdb_memory_limit} | RF: n_jobs={rf_n_jobs} | LGB: {lgb_threads} threads")

#### Librerías y Path Variables

In [ ]:
# Importando Librerías

import os, sys
os.environ['PATH'] = os.path.join(sys.prefix, 'Library', 'bin') + os.pathsep + os.environ['PATH']
import duckdb

import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.express as px

In [ ]:
# Paths Variables

from pathlib import Path
nb_file = globals().get('__vsc_ipynb_file__')                 # CAMBIO: ruta de la notebook (la define VS Code)
cwd = Path(nb_file).parent if nb_file else Path.cwd()         # CAMBIO: carpeta de la notebook; si no es VS Code, usa el cwd
RAIZ = (cwd / '../../../../05_ExpMeses').resolve()

sys.path.insert(0, str(cwd))                                  # semillas.py está al lado de la notebook
from semillas import SEMILLAS as semillas

base_path = RAIZ.as_posix() + '/'                             # as_posix: barras normales también en Windows
print(RAIZ, RAIZ.exists(), (cwd / 'semillas.py').exists())

dataset_path = base_path + 'data/'
db_path = base_path + 'databases/'
modelos_path = base_path + 'modelos/'
out_path = base_path + 'out/'
dataset_file = 'competencia_02_crudo.csv.gz'
ruta_crudo = dataset_path + dataset_file
print(ruta_crudo, Path(ruta_crudo).exists())

#### Conversión del csv.gz original --> parquet

In [ ]:
%%time
# Conversión única: csv.gz original -> parquet (mismo contenido, todos los meses y columnas)
ruta_parquet = dataset_path + 'competencia_02_crudo.parquet'

if Path(ruta_parquet).exists():
    print('El parquet ya existe, no se regenera:', ruta_parquet)
else:
    con_conv = duckdb.connect()
    con_conv.execute(f"SET threads = {duckdb_threads}")
    con_conv.execute(f"SET memory_limit = '{duckdb_memory_limit}'")
    con_conv.execute(f"""
        COPY (SELECT * FROM read_csv('{ruta_crudo}', sample_size = -1))
        TO '{ruta_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print('listo:', round(Path(ruta_parquet).stat().st_size / 1e6), 'MB')

# Columnas que quedaron como texto (las 0/1 como tmobile_app hay que pasarlas a número)
con_chk = duckdb.connect()
print('Columnas de texto:', [r[0] for r in con_chk.execute(f"SELECT column_name FROM (DESCRIBE SELECT * FROM read_parquet('{ruta_parquet}')) WHERE column_type = 'VARCHAR'").fetchall()])

#### Carga y Análisis del Dataset Completo

In [ ]:
%%time
con = duckdb.connect()
con.execute(f"SET threads = {duckdb_threads}")
con.execute(f"SET memory_limit = '{duckdb_memory_limit}'")
con.execute("SET enable_progress_bar = false")
con.execute(f"CREATE OR REPLACE TABLE dcrudo AS SELECT * FROM read_parquet('{ruta_parquet}')")
print(con.execute("SELECT count(*) FROM dcrudo").fetchall()[0][0], 'filas')

#### Filas y Clientes por Mes

In [ ]:
mes_resumen = con.execute("""
    SELECT foto_mes,
           count(*)                                      AS filas,
           count(DISTINCT numero_de_cliente)             AS clientes,
           count(*) - count(DISTINCT numero_de_cliente)  AS duplicados      -- tiene que dar 0
    FROM dcrudo
    GROUP BY foto_mes
    ORDER BY foto_mes
""").df()
print(len(mes_resumen), 'meses |', con.execute("SELECT count(DISTINCT numero_de_cliente) FROM dcrudo").fetchall()[0][0], 'clientes distintos')
mes_resumen

In [ ]:
meses_es = ['ene', 'feb', 'mar', 'abr', 'may', 'jun', 'jul', 'ago', 'sep', 'oct', 'nov', 'dic']
g = mes_resumen.assign(mes=pd.to_datetime(mes_resumen['foto_mes'].astype(str), format='%Y%m'))
g['etiqueta'] = g['mes'].dt.month.map(lambda m: meses_es[m - 1]) + '-' + g['mes'].dt.year.astype(str)     # ene-2019

fig = px.line(g, x='mes', y='clientes', custom_data=['etiqueta'], title='Clientes por mes')
fig.update_traces(line=dict(color='#2a78d6', width=2),
                  hovertemplate='%{customdata[0]}: %{y:,.0f} clientes<extra></extra>')
fig.update_layout(template='plotly_white', separators=',.',
                  xaxis=dict(title=None, tickmode='array', tickvals=g['mes'], ticktext=g['etiqueta'], tickangle=-90),
                  yaxis=dict(title='Clientes', tickformat=',d'),
                  height=420)
fig.show()

#### Columnas y Tipos

In [ ]:
esquema = con.execute("DESCRIBE dcrudo").df()[['column_name', 'column_type']]
print(len(esquema), 'columnas | clase_ternaria está:', 'clase_ternaria' in esquema['column_name'].values)
print('Columnas de texto:', list(esquema.loc[esquema['column_type'] == 'VARCHAR', 'column_name']))
esquema['column_type'].value_counts()

#### Creación Clase Ternaria (solo 2019, nov-19 solo baja+1 y dic-19 null)

In [ ]:
con.close()   # para liberar memoria, ya que la notebook es muy pesada y a veces se cuelga en la ejecución de celdas posteriores

In [ ]:
%%time
MES_DESDE, MES_HASTA = 201901, 201912        # meses con clase -- el resto del dataset queda sin clase

con_m = duckdb.connect()
con_m.execute(f"SET threads = {duckdb_threads}")
con_m.execute(f"SET memory_limit = '{duckdb_memory_limit}'")
con_m.execute("SET enable_progress_bar = false")

con_m.execute(f"""
    CREATE OR REPLACE TABLE competencia_02_meses AS
    WITH crudo AS (
        SELECT numero_de_cliente, foto_mes
        FROM read_parquet('{ruta_parquet}')
        WHERE foto_mes BETWEEN {MES_DESDE} AND {MES_HASTA}
    ), periodos AS (
        SELECT DISTINCT foto_mes FROM crudo
    ), clientes AS (
        SELECT DISTINCT numero_de_cliente FROM crudo
    ), todo AS (
        SELECT numero_de_cliente, foto_mes FROM clientes CROSS JOIN periodos
    ), con_clase AS (
        SELECT
            t.numero_de_cliente
            , t.foto_mes
            , IF(c.numero_de_cliente IS NULL, 0, 1) AS mes_0
            , LEAD(mes_0, 1) OVER (PARTITION BY t.numero_de_cliente ORDER BY foto_mes) AS mes_1
            , LEAD(mes_0, 2) OVER (PARTITION BY t.numero_de_cliente ORDER BY foto_mes) AS mes_2
            , CASE
                WHEN mes_1 = 0 THEN 'BAJA+1'
                WHEN mes_2 = 0 THEN 'BAJA+2'
                WHEN mes_1 IS NULL OR mes_2 IS NULL THEN NULL
                ELSE 'CONTINUA'
              END AS clase_ternaria
        FROM todo t
        LEFT JOIN crudo c USING (numero_de_cliente, foto_mes)
    ), clase AS (
        SELECT numero_de_cliente, foto_mes, clase_ternaria FROM con_clase WHERE mes_0 = 1
    )
    SELECT p.*, k.clase_ternaria
    FROM read_parquet('{ruta_parquet}') p
    LEFT JOIN clase k USING (numero_de_cliente, foto_mes)
""")

##### Estadísticas

In [ ]:
# 1) Mismas filas que el crudo
n_crudo = con_m.execute(f"SELECT count(*) FROM read_parquet('{ruta_parquet}')").fetchall()[0][0]
n_meses = con_m.execute("SELECT count(*) FROM competencia_02_meses").fetchall()[0][0]
print(n_crudo, n_meses, 'mismas filas:', n_crudo == n_meses)

# 2) Clientes por clase y mes
con_m.execute("""
    PIVOT (SELECT foto_mes, coalesce(clase_ternaria, 'SIN_CLASE') AS clase_ternaria, numero_de_cliente
           FROM competencia_02_meses)
    ON clase_ternaria USING count(numero_de_cliente) GROUP BY foto_mes ORDER BY foto_mes
""").df()

In [ ]:
# 3) Proporción de cada clase
con_m.execute("""
    SELECT coalesce(clase_ternaria, 'SIN_CLASE') AS clase_ternaria
           , count(*) AS cantidad
           , round(100.0 * count(*) / sum(count(*)) OVER (), 3) AS porcentaje
    FROM competencia_02_meses
    GROUP BY clase_ternaria
    ORDER BY clase_ternaria
""").df()

##### Guardado

In [ ]:
ruta_meses = dataset_path + 'competencia_02_meses.parquet'
con_m.execute(f"""
    COPY (SELECT * FROM competencia_02_meses ORDER BY foto_mes, numero_de_cliente)
    TO '{ruta_meses}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print('guardado:', ruta_meses, round(Path(ruta_meses).stat().st_size / 1e6), 'MB')
con_m.close()